# 05. Pruning, Memoization and Complexity

Raw backtracking can be too slow. Two upgrades: **prune** hopeless branches, and **memoize** repeated states (which leads to Dynamic Programming).

**Learning order:** PRUNING IDEAS → K EQUAL SUBSETS → MATCHSTICKS → MEMO: TARGET SUM → MEMO: WORD BREAK → WHEN TO SWITCH TO DP → INTERVIEW THINKING

## 1. Pruning ideas

- **Bounds:** stop if the remaining items cannot reach the goal.
- **Sort + break:** when a value is too big, all later values are too.
- **Symmetry:** if two choices are equivalent (two empty buckets), try only one.
- **Order:** try the hardest/biggest items first (they fail faster).
- **Early exit:** return as soon as the first valid answer is found (decision problems).

## 2. Partition to K Equal Sum Subsets

Can the array be split into `k` groups with equal sum? Each group sum = `total / k`. Place numbers **one by one into buckets**.

Prunes: sort **descending**; skip a bucket with the same sum as one already tried; if a number does not fit into an **empty** bucket, it will not fit anywhere → stop.

In [1]:
def can_partition_k(nums, k):
    total = sum(nums)
    if total % k or max(nums) > total // k:
        return False
    target = total // k
    nums.sort(reverse=True)
    buckets = [0] * k
    def backtrack(i):
        if i == len(nums):
            return True
        for b in range(k):
            if buckets[b] + nums[i] <= target:
                buckets[b] += nums[i]
                if backtrack(i + 1):
                    return True
                buckets[b] -= nums[i]
            if buckets[b] == 0:                  # symmetry prune: other empty buckets are identical
                break
        return False
    return backtrack(0)

print(can_partition_k([4, 3, 2, 3, 5, 2, 1], 4), can_partition_k([1, 2, 3, 4], 3))   # True False
print(can_partition_k([1, 1, 2, 2, 2], 4))   # True  (Matchsticks to Square: four sides of length 2)

True False
True


## 3. Memoization: Target Sum

Assign `+` or `-` to every number to reach `target`. Plain backtracking explores 2ⁿ paths. But the state `(index, current sum)` **repeats**. Cache it with `lru_cache`: the answer for the same state is always the same. Now at most `n × (range of sums)` states.

In [2]:
from functools import lru_cache

def find_target_sum_ways(nums, target):
    @lru_cache(maxsize=None)
    def count(i, total):
        if i == len(nums):
            return 1 if total == target else 0
        return count(i + 1, total + nums[i]) + count(i + 1, total - nums[i])
    return count(0, 0)

print(find_target_sum_ways([1, 1, 1, 1, 1], 3))   # 5

5


## 4. Memoization: Word Break

Can `s` be split into dictionary words? Try every prefix that is a word, recurse on the rest. The state is just **the start index**, so memoization makes it O(n²) instead of exponential.

In [3]:
def word_break(s, word_dict):
    words = set(word_dict)
    @lru_cache(maxsize=None)
    def can(start):
        if start == len(s):
            return True
        return any(s[start:end] in words and can(end) for end in range(start + 1, len(s) + 1))
    return can(0)

print(word_break("leetcode", ["leet", "code"]), word_break("catsandog", ["cats", "dog", "sand", "and", "cat"]))   # True False

True False


## 5. When to switch to DP

Draw the recursion tree. If **the same subproblem appears many times** and the state is small (index, remaining sum, mask), add memoization → **top-down DP**. If the question asks to **list** all solutions, memoization cannot remove the output cost, so you still backtrack.

| Question | Tool |
|---|---|
| return all valid arrangements | backtracking |
| can it be done? how many ways? best value? with repeated states | memoization / DP |
| choices are greedy-provable | greedy |

## 6. Interview thinking

**Say:** "Naive search is exponential. I prune by ... and if states repeat I add a memo keyed by (...) to get O(states × transitions)."

**Common mistakes:** memoizing a function that depends on mutable global state, putting a list (unhashable) in the memo key, forgetting the symmetry prune (TLE), exploring before checking the quick necessary conditions (`total % k`).

## Quick Revision

- Prune with bounds, sort + break, symmetry, quick feasibility checks.
- Repeated `(index, remaining)` states → `lru_cache`.
- Listing all answers → backtracking. Counting/feasibility → memo/DP.
- Check necessary conditions first (`total % k`, `max > target`).